# 01 — Problem, data, and findings

This notebook establishes the problem before we choose features or models: **what data and signals were we given, and why can they not be copied directly into H/A/D labels?**

The notebook is descriptive.

- Each section explains one signal.
- The next code cell checks it against our local files.
- No model is trained here.
- Public labels are used only for a small descriptive sanity check; they are not used for tuning.

## The problem in one sentence

For every tactical frame, classify the match state as Home possession (`H`), Away possession (`A`), dead ball (`D`), or outside the scored intervals (`X`). The difficult part is not calculating a distance; it is distinguishing a genuine controlled reception from a one-frame touch, a contested ball, or a ball in flight.

## Notebook map

```text
1. ATD tracking       → players and ball
2. Homography         → pitch coordinates ↔ image pixels
3. Smart Tagging      → automatic football-phase context
4. Merged frame table → small model-ready evidence table
```

The public sanity window is `11,185–18,684`.

Later examples use middle frames from the graded halves.
Their labels are hidden.

In [ ]:
# Purpose: load the local files and define the paths used by every evidence section.
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'possession' else Path.cwd().resolve()
DATA = ROOT / 'data'
TACTICAL = DATA / 'tactical'
sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)
print('Tactical files:')
for path in sorted(TACTICAL.glob('*')):
    print(f'  {path.name}: {path.stat().st_size / 1024**2:.1f} MB')
print('Submission frames: 0..199318 (199,319 rows)')

## 1. ATD — Automatic Tracking Data

ATD means **Automatic Tracking Data**.

It estimates: 
- the ball position;
- player-track positions;
- supplied movement values.

The coordinates are on the pitch, not camera pixels.

### ATD column glossary

| Column pattern | Data given | Glossary / meaning | How we may use it |
|---|---|---|---|
| `period_id` | period number | Which recording period the row belongs to | Separate match periods |
| `timestamp` | time value | Time associated with the frame | Order and time-based features |
| `frame_id` | integer `1..199319` | ATD's 1-based frame counter | Convert to submission frame with `frame_id - 1` |
| `ball_state` | provider field | Ball-state field from the tracking export | Inspect, but check missingness first |
| `ball_owning_team_id` | provider team ID | Provider's ball-owner field | Inspect, but it is not our final `H/A/D` label |
| `ball_x`, `ball_y` | number or missing | Ball position in normalised pitch coordinates, not image pixels | Ball location and visibility |
| `ball_z` | number or missing | Ball height coordinate, if supplied | Detect airborne balls if populated |
| `ball_speed` | number or missing | Ball speed field, if supplied | Ball movement features |
| `k_x` | number or missing | x-position of temporary track slot `k` in normalised pitch coordinates, not image pixels | Player location |
| `k_y` | number or missing | y-position of temporary track slot `k` in normalised pitch coordinates, not image pixels | Player location |
| `k_d` | partially populated here | Parsed third slot field; its semantics are not relied on by the baseline | Treat as optional until separately validated |
| `k_s` | number or missing | Supplied speed for slot `k` | Movement features |

Example: `44_x`, `44_y`, `44_d`, `44_s` describe temporary **track slot 44**.

They do not mean player number 44.

- x/y are normalised pitch coordinates, approximately `0..1`;
- x/y are not image pixels;
- `_d` is only partially populated in this export and is not relied on by the baseline;
- the XML describes speed as km/h.

In [ ]:
# Purpose: load ATD and show its shape, frame convention, sample columns, and missingness.
from kloppy import metrica

atd_dataset = metrica.load_tracking_epts(
    meta_data=TACTICAL / 'BAR-ATH_tactical_atd.xml',
    raw_data=TACTICAL / 'BAR-ATH_tactical_atd.txt',
)
atd = atd_dataset.to_df()

print('Shape:', atd.shape)
print('Frame IDs:', atd.frame_id.min(), 'to', atd.frame_id.max())
print('Submission frames:', atd.frame_id.min() - 1, 'to', atd.frame_id.max() - 1)

core_columns = ['period_id', 'timestamp', 'frame_id', 'ball_state', 'ball_owning_team_id', 'ball_x', 'ball_y', 'ball_z', 'ball_speed']
track_columns = ['1_x', '1_y', '1_d', '1_s', '44_x', '44_y', '44_d', '44_s', '46_x', '46_y', '46_d', '46_s']
available_columns = [column for column in core_columns + track_columns if column in atd.columns]
print('\nReadable sample:')
print(atd[available_columns].head(3).to_string(index=False))

print('\nMissingness of important fields:')
missingness = atd[available_columns].isna().mean().mul(100).round(1)
print(missingness.to_string())

## 2. Homography — the camera-to-pitch map

A camera views the pitch at an angle.

The flat pitch is therefore distorted in the image.

A Homography is the mathematical mapping from pitch coordinates to that camera's pixels.

There are two files:

- **Tactical** → aligned to the submission timeline; used below.
- **Broadcast** → paired with broadcast ATD/video; not synchronised.

Broadcast use requires a separate alignment step.

```text
ATD pitch position (x, y) → video pixel position (u, v)
```

### Homography column glossary

| Column | Data given | Glossary / meaning |
|---|---|---|
| `frame` | integer `0..199318` | 0-based tactical video frame |
| `h00..h22` | 9 numbers | The 3×3 transform matrix, stored row by row |
| `reprojection_error` | number | How far projected reference points miss their expected image locations; lower is generally better |
| `valid` | `0` or `1` | Whether the geometry estimate is marked usable |

Matrix layout:

    [h00 h01 h02]
    [h10 h11 h12]
    [h20 h21 h22]

Key distinction:

- ATD gives normalised pitch coordinates.
- Homography converts them to image pixels.
- Homography does not identify possession.
- It helps project tracking positions onto video.

### Do we need pixel positions?

Not for every model. ATD pitch coordinates are enough for:

- distances between players and the ball;
- team shape and spacing;
- nearest-player and pressure features.

Pixel positions help when we want to:

- draw ATD locations on the video;
- inspect whether a tracking point looks correct;
- combine tracking with image-based features, such as a crop or visual detector.

So Homography is an optional bridge from pitch-space data to camera-image data. It does not add a new player location; it expresses the same location in a different coordinate system.

In [ ]:
# Purpose: load Homography and verify the matrix columns, frame range, validity, and error.
homography = pd.read_csv(TACTICAL / 'BAR-ATH_tactical_homography.csv')
matrix_columns = ['h00', 'h01', 'h02', 'h10', 'h11', 'h12', 'h20', 'h21', 'h22']

print('Shape:', homography.shape)
print('Frame IDs:', homography.frame.min(), 'to', homography.frame.max())
print('All nine matrix columns present:', set(matrix_columns).issubset(homography.columns))
print('Valid proportion:', round(homography.valid.mean(), 4))
print('Mean reprojection error:', round(homography.reprojection_error.mean(), 4))
print('\nReadable sample:')
print(homography[['frame', *matrix_columns, 'reprojection_error', 'valid']].head(3).to_string(index=False))

## 3. Smart Tagging — automatic football-phase context

Smart Tagging is automatic football-analysis context.

It marks intervals that look like phases or events,
such as restarts and set pieces.

### Smart Tagging column glossary

| Field | Data given | Glossary / meaning |
|---|---|---|
| `code` | text | Broad event/phase category, such as `SET PIECES` |
| `start_s` / `end_s` | seconds | Start and end time of the tagged interval |
| `start_frame` / `end_frame` | frame IDs | Start and end of the tagged interval on the tactical timeline |
| `tags` | dictionary | More detailed context, such as team, type, or half |

Use it to ask:

> What type of football phase may be happening?

It is not the official possession label.
Treat it as context and expect some errors.

In [ ]:
# Purpose: load Smart Tagging and show event counts plus one parsed event.
from starter.load_smart_tagging import load as load_smart_tagging

events = load_smart_tagging()
print('Number of events:', len(events))
print('Event codes:')
print(pd.Series([event['code'] for event in events]).value_counts().to_string())
print('\nOne event example:')
print(events[0])

## Finding: Smart Tagging is context, not frame-level ground truth

We use the public five-minute labels here only as a descriptive sanity check. The question is not whether Smart Tagging is useless; it is whether one tagged interval can be copied directly to every frame.

The next cell measures overlap for `SET PIECES` in the public window and shows one public-window trajectory plot. The plot is evidence for the problem: a tag spans a time interval, while the ball visibility, player geometry, and true `D`/H/A labels can change within that interval.

Operational conclusion → use Smart Tagging to open a scenario window; use ATD geometry and controlled-reception evidence to make the frame-level decision.

In [ ]:
# Purpose: quantify Smart Tagging's public-window coverage and plot one public ball gap.
from possession.baseline_pipeline import ClosestTeamRuleModel, slot_team_map
from possession.calculations.plotting import plot_trajectory_window, select_representative_ball_gap
from games import asset_paths
from starter.load_smart_tagging import load as load_smart_tagging
import matplotlib.pyplot as plt

public_gt = pd.read_csv(ROOT / 'evaluation' / 'gt' / 'BAR-ATH_possession_gt_public.csv')
public_mask = public_gt['frame'].between(11_185, 18_684)
set_piece_mask = np.zeros(len(public_gt), dtype=bool)
for event in load_smart_tagging():
    if event['code'] == 'SET PIECES':
        left = max(0, int(event['start_frame']))
        right = min(len(public_gt) - 1, int(event['end_frame']))
        set_piece_mask[left:right + 1] = True
set_piece_public = set_piece_mask & public_mask.to_numpy()
public_d = public_mask.to_numpy() & public_gt['label'].eq('D').to_numpy()
print('Public SET PIECES frames:', int(set_piece_public.sum()))
print('Labels inside those frames:', public_gt.loc[set_piece_public, 'label'].value_counts().to_dict())
print('D precision inside SET PIECES:', round(float(public_gt.loc[set_piece_public, 'label'].eq('D').mean()), 3))
print('Public D coverage by SET PIECES:', round(float((set_piece_public & public_d).sum() / public_d.sum()), 3))

public_features, _ = ClosestTeamRuleModel().build_features()
public_features = public_features.loc[public_features['frame'].between(11_185, 18_684)].copy()
gap = select_representative_ball_gap(public_features)
if gap is not None:
    gap_start, gap_end = gap
    team_by_slot = slot_team_map(asset_paths()['atd_meta'])
    fig, ax = plot_trajectory_window(public_features, max(11_185, gap_start - 25), min(18_684, gap_end + 25), team_by_slot=team_by_slot)
    plt.show()
else:
    print('No internal ball gap found in the public window.')


## 4. A small merged frame-level table

We merge signals using the tactical video frame.

This table is small and interpretable.
It is not yet the final model feature table.

```text
ATD          → ball/player positions on the normalised pitch
Homography   → converts those pitch coordinates → video pixels
Smart Tagging→ football-phase context
                 ↓
          one row per tactical frame
```

In [ ]:
# Purpose: align ATD and Homography into one compact row-per-frame evidence table.
atd_frame = atd.copy()
atd_frame['frame'] = atd_frame['frame_id'] - 1
ball_x = 'ball_x' if 'ball_x' in atd_frame.columns else None
ball_y = 'ball_y' if 'ball_y' in atd_frame.columns else None
player_x = [column for column in atd_frame.columns if str(column).endswith('_x') and 'ball' not in str(column).lower()]

features = pd.DataFrame({'frame': atd_frame['frame']})
features['ball_visible'] = atd_frame[[ball_x, ball_y]].notna().all(axis=1)
features['visible_track_count'] = atd_frame[player_x].notna().sum(axis=1)

geometry = homography[['frame', 'valid', 'reprojection_error']].rename(columns={'valid': 'homography_valid'})
merged = features.merge(geometry, on='frame', how='left')

sample_frames = [11_185, 11_186, 50_918, 50_919, 150_502, 150_503]
print('Merged shape:', merged.shape)
print('Sample rows from public sanity + graded middle windows:')
print(merged[merged.frame.isin(sample_frames)].to_string(index=False))
print('\nCompact aggregate facts:')
print(merged[['ball_visible', 'visible_track_count', 'homography_valid', 'reprojection_error']].describe().round(3).to_string())

## 5. Availability on graded frames

Global missingness is not enough: a field may exist outside the scored windows but be unavailable where the score is calculated. We therefore compare graded and non-graded frames.

- Graded: `11,185–90,651` and `110,808–190,197`.
- Drop fields that are null for every graded frame.
- Keep partially missing fields, but expose visibility/missingness explicitly.
- This is evidence for modelling decisions, not score tuning.

In [ ]:
# Purpose: compare ATD missingness on graded versus non-graded frames.
from possession.baseline_pipeline import graded_mask

graded = graded_mask(len(atd_frame))
availability_rows = []
for column in atd_frame.columns:
    if column == 'frame':
        continue
    availability_rows.append({
        'column': column,
        'graded_missing_%': round(100 * atd_frame.loc[graded, column].isna().mean(), 1),
        'non_graded_missing_%': round(100 * atd_frame.loc[~graded, column].isna().mean(), 1),
    })
availability = pd.DataFrame(availability_rows).sort_values(['graded_missing_%', 'column'], ascending=[False, True])
print(f'Graded frames: {graded.sum():,}; non-graded frames: {(~graded).sum():,}')
print('Always missing on graded frames:')
print(availability[availability['graded_missing_%'] == 100].to_string(index=False))
print('Most incomplete but not always-missing fields on graded frames:')
print(availability[(availability['graded_missing_%'] > 0) & (availability['graded_missing_%'] < 100)].head(20).to_string(index=False))

## 6. Frame-to-match-time index

This creates a human-checkable time column for the graded frames. `video_time_s` is exact on the tactical timeline at 25 fps. `half_elapsed_s` is measured from the verified kickoff frame and can be negative for the graded second-half pre-kickoff interval.

`match_clock_s` is a kickoff-anchored analytical clock: 0:00 at first-half kickoff and 45:00 at second-half kickoff. It is not a claim about the referee's displayed stoppage-time clock.

In [ ]:
# Purpose: create a graded-only frame index for checking the tactical data against the match.
import importlib
import possession.baseline_pipeline as pipeline
pipeline = importlib.reload(pipeline)

time_index = pipeline.add_match_time_columns(atd_frame[['frame']].copy())
time_index['graded'] = pipeline.graded_mask(len(time_index))
graded_time_index = time_index[time_index['graded']].copy()
graded_time_index['video_time'] = pd.to_timedelta(graded_time_index['video_time_s'], unit='s')
graded_time_index['match_clock'] = graded_time_index['match_clock_s'].map(lambda value: f'{int(value // 60):02d}:{value % 60:05.2f}' if value >= 0 else f'pre-kickoff ({value:.2f}s)')
print(graded_time_index[['frame', 'match_half', 'video_time', 'half_elapsed_s', 'match_clock', 'match_clock_status']].head(3).to_string(index=False))
print('\nLast graded frames in each block:')
print(graded_time_index.groupby('match_half', sort=False).tail(3)[['frame', 'match_half', 'match_clock', 'match_clock_status']].to_string(index=False))

## What we know now

- ATD gives positions and speeds, not the final possession answer.
- Track slots are temporary; `_d` is empty here.
- Homography connects pitch coordinates to camera pixels.
- Smart Tagging gives phase context, not ground truth.
- Next: team-aware distances, ball gaps, dead-ball context, and history.